# Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from keras.callbacks import TensorBoard, ModelCheckpoint
from tensorflow.keras.layers import Input, Conv1D, Layer, Dropout, Bidirectional, LSTM, Dense, Add, LayerNormalization, GlobalAveragePooling1D, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Dataset

In [ ]:
btc = pd.read_csv("dataset/BTCUSD_1m_Binance.csv")
eth = pd.read_csv("dataset/ETHUSD_1m_Binance.csv")

cols_to_drop = ["High", "Low", "Close", "Close time"]
btc = btc.drop(columns=cols_to_drop, errors='ignore')
eth = eth.drop(columns=cols_to_drop, errors='ignore')


btc = btc.rename(columns={col: f"{col} BTC" for col in btc.columns if col != "Open time"})
eth = eth.rename(columns={col: f"{col} ETH" for col in eth.columns if col != "Open time"})

merged_df = pd.merge(btc, eth, on="Open time", how="inner")
merged_df = merged_df.sort_values(by="Open time").reset_index(drop=True)

merged_df.to_csv("dataset/merged_BTC_ETH.csv", index=False)

print(merged_df.head())

# Prétraitement des Données

In [ ]:
merged_df['Open time'] = pd.to_datetime(merged_df['Open time'], format='%Y-%m-%d %H:%M:%S')
display(merged_df.info())

# Visualisation

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(merged_df['Open time'], merged_df['Open BTC'], label='BTC Open Price', color='blue')
plt.title('BTC Open Prices Over Year')
plt.xlabel('Year')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(merged_df['Open time'], merged_df['Open ETH'], label='ETH Open Price', color='orange')
plt.title('ETH Open Prices Over Year')
plt.xlabel('Year')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

# Callback

In [ ]:
callbacks = [
    TensorBoard(log_dir='./logs', histogram_freq=1, write_graph=True),
    ModelCheckpoint(filepath='model_lstm.keras', save_best_only=True),
]

# Data Split

In [ ]:
train_data = merged_df.iloc[:int(len(merged_df) * 0.8)].copy()
test_data = merged_df.iloc[int(len(merged_df) * 0.8):].copy()

features = [col for col in merged_df.columns if col != 'Open time']
target = 'Open ETH'

scaler_features = MinMaxScaler()
scaler_target = MinMaxScaler()

train_features_scaled = scaler_features.fit_transform(train_data[features])
train_target_scaled = scaler_target.fit_transform(train_data[[target]])

test_features_scaled = scaler_features.transform(test_data[features])
test_target_scaled = scaler_target.transform(test_data[[target]])

look_back = 10

train_generator = TimeseriesGenerator(train_features_scaled, train_target_scaled, length=look_back, batch_size=32)

test_generator = TimeseriesGenerator(test_features_scaled, test_target_scaled, length=look_back, batch_size=32)

# CBAM

In [ ]:
class CBAM(Layer):
    def __init__(self, ratio=8, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.kernel_size = kernel_size

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden = max(channels // self.ratio, 1)

        self.gap = GlobalAveragePooling1D()
        self.gmp = GlobalMaxPooling1D()
        self.fc1 = Dense(hidden, activation='relu', kernel_initializer='he_normal', use_bias=True)
        self.fc2 = Dense(channels, activation=None, kernel_initializer='he_normal', use_bias=True)

        self.spatial_conv = Conv1D(filters=1, kernel_size=self.kernel_size,
                                   padding='same', activation='sigmoid',
                                   kernel_initializer='he_normal')
        super().build(input_shape)

    def call(self, inputs):
        avg_pool = self.fc2(self.fc1(self.gap(inputs)))
        max_pool = self.fc2(self.fc1(self.gmp(inputs)))
        channel_attn = tf.nn.sigmoid(avg_pool + max_pool)
        channel_attn = tf.reshape(channel_attn, (-1, 1, tf.shape(inputs)[-1]))
        x = inputs * channel_attn

        avg_spatial = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_spatial = tf.reduce_max(x, axis=-1, keepdims=True)
        spatial = tf.concat([avg_spatial, max_spatial], axis=-1)
        spatial_attn = self.spatial_conv(spatial)

        return x * spatial_attn

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"ratio": self.ratio, "kernel_size": self.kernel_size})
        return cfg

# Model

In [ ]:
inputs = Input(shape=(look_back, len(features)))

x = Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(inputs)
x = Dropout(0.1)(x)

residual = x

x_main = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(x)
x_main = CBAM(ratio=8, kernel_size=7)(x_main)
x_main = Dropout(0.2)(x_main)

residual_projected = Conv1D(filters=64, kernel_size=1, padding='same')(residual)

x = Add()([x_main, residual_projected])
x = LayerNormalization()(x)

x = Bidirectional(LSTM(units=100, return_sequences=True))(x)
x = Dropout(0.2)(x)

x = Bidirectional(LSTM(units=50, return_sequences=False))(x)
x = Dropout(0.2)(x)

x = Dense(50, activation='relu')(x)
x = Dropout(0.2)(x)

x = Dense(25, activation='relu')(x)

outputs = Dense(1)(x)

model_crypto_residual = Model(inputs=inputs, outputs=outputs)
model_crypto_residual.compile(optimizer='adam', loss='huber', metrics=['mean_absolute_error', 'mean_squared_error'])

In [ ]:
model_training = model_crypto_residual.fit(train_generator, epochs=20, verbose=1, validation_data=test_generator, callbacks=callbacks)
model_pred = model_crypto_residual.predict(test_generator)